# MCD64A1 burned area per municipality of Santa Catarina, 2012-2024

**FINN_SC · `gee/mcd64a1_municipal_2012_2024.ipynb`** (co-authors' review of 2026-10-19, comment C11)

Second independent burned-area reference for the comparison of section 2.7 of the manuscript, so that the difference
between the FINN area and the mapped area does not rest on MapBiomas Fogo alone.

* Product: MODIS Collection 6.1 burned area, Earth Engine collection `MODIS/061/MCD64A1`, band `BurnDate` (day of burn,
  0 = unburned), monthly, ~500 m (Giglio et al., 2018, *Remote Sensing of Environment* 217, 72-85).
* Annual burned mask: a pixel is burned in a year if `BurnDate > 0` in any month of that year (one count per pixel
  and year, as the MapBiomas Fogo annual layer).
* Area: mask × `ee.Image.pixelArea()`, summed over each municipal polygon of the IBGE 2024 mesh (same mesh as the
  FINN and MapBiomas tables) in the native projection of the product.
* Output: `data/external/mcd64a1_SC_municipal_km2_2012_2024.csv` (CD_MUN, NM_MUN, year, burned_km2), then the same
  comparison statistics as in the manuscript (ratio of the areas, correlation of the state series, municipal Spearman
  correlation per year, share of the Serrana).

The 500 m product misses more of the small burns than the 30 m MapBiomas map (Giglio et al., 2018; Alencar et al.,
2022), so its area is expected to be smaller than the MapBiomas area in Santa Catarina.

In [ ]:
# 1. Packages (skip if already installed)
%pip install -q earthengine-api geemap geopandas pandas scipy

In [ ]:
# 2. Parameters
from pathlib import Path
# repository root = parent of gee/ (local run); otherwise the current folder (e.g. Colab, with the input files uploaded)
REPO_ROOT = Path.cwd().resolve().parent
if not (REPO_ROOT / "config" / "config.yml").is_file():
    REPO_ROOT = None
PROJECT = "ee-jessicajcss"   # <-- Earth Engine Cloud project (change it to your own)
YEARS = list(range(2012, 2025))
OUT_NAME = "mcd64a1_SC_municipal_km2_2012_2024.csv"
OUT = (REPO_ROOT / "data" / "external" / OUT_NAME) if REPO_ROOT else Path.cwd() / OUT_NAME
print("output:", OUT)

In [ ]:
# 3. Earth Engine authentication
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine initialised, project", PROJECT)

In [ ]:
# 4. Municipal polygons of Santa Catarina: IBGE 2024 mesh, the same used for the FINN and MapBiomas Fogo tables.
#    Earth Engine receives the simplified copy gee/SC_municipios_2024_simplified.gpkg (tools/simplify_municipios_for_gee.py:
#    shared borders simplified at 0.0005 degrees, about 55 m; area change per municipality <= 0.07 %), because the full
#    shapefile (about 70 MB as GeoJSON) exceeds the 10 MB limit of an Earth Engine request.
#    In Colab, this cell asks for that one file (folder gee/ of the repository, about 2.7 MB).
import geopandas as gpd, geemap, pandas as pd
GPKG = "SC_municipios_2024_simplified.gpkg"
if REPO_ROOT is not None:
    gpkg_path = REPO_ROOT / "gee" / GPKG
else:
    gpkg_path = Path.cwd() / GPKG
    if not gpkg_path.is_file():
        try:
            from google.colab import files
            print(f"Select the file {GPKG} (folder gee/ of the repository)")
            for name in files.upload():
                if name != GPKG and name.startswith("SC_municipios_2024_simplified"):
                    Path(name).rename(GPKG)          # Colab renames a repeated upload, e.g. "... (1).gpkg"
        except ImportError:
            pass
    assert gpkg_path.is_file(), f"copy {GPKG} (folder gee/ of the repository) to {Path.cwd()}"
mun_s = gpd.read_file(gpkg_path)
mun_s["CD_MUN"] = mun_s["CD_MUN"].astype("int64")
assert len(mun_s) == 295 and mun_s.CD_MUN.is_unique and mun_s.mesorregiao.notna().all(), len(mun_s)
if REPO_ROOT is not None:   # full-resolution mesh for the local steps
    mun = gpd.read_file(REPO_ROOT / "data" / "inputs" / "SC_Municipios_2024.shp", encoding="cp1252").to_crs(4326)
    mun["CD_MUN"] = mun["CD_MUN"].astype("int64")
    mun = mun.merge(mun_s[["CD_MUN", "mesorregiao"]], on="CD_MUN", how="left")
else:
    mun = mun_s
mun_ee = geemap.geopandas_to_ee(mun_s[["CD_MUN", "NM_MUN", "mesorregiao", "geometry"]])
print(len(mun_s), "municipalities;", mun_s.groupby("mesorregiao").size().to_dict())


In [ ]:
# 5. Annual burned area per municipality (km2)
col = ee.ImageCollection("MODIS/061/MCD64A1").select("BurnDate")
proj = col.first().projection()
rows = []
for y in YEARS:
    yc = col.filterDate(f"{y}-01-01", f"{y + 1}-01-01")
    n_img = yc.size().getInfo()
    burned = yc.map(lambda im: im.gt(0)).max().unmask(0)          # burned in any month of the year
    area = burned.multiply(ee.Image.pixelArea()).divide(1e6).rename("burned_km2").setDefaultProjection(proj)
    fc = area.reduceRegions(collection=mun_ee, reducer=ee.Reducer.sum(), crs=proj, scale=proj.nominalScale().getInfo(), tileScale=4)
    d = geemap.ee_to_df(fc.select(["CD_MUN", "NM_MUN", "sum"], retainGeometry=False))
    d = d.rename(columns={"sum": "burned_km2"})
    d["year"] = y
    rows.append(d)
    print(y, "images:", n_img, "| state total (km2):", round(d.burned_km2.sum(), 1))
mcd = pd.concat(rows, ignore_index=True)[["CD_MUN", "NM_MUN", "year", "burned_km2"]]
mcd["CD_MUN"] = mcd.CD_MUN.astype("int64")
assert len(mcd) == 295 * len(YEARS)

In [ ]:
# 6. Save and record the SHA-256
import hashlib
OUT.parent.mkdir(parents=True, exist_ok=True)
mcd.sort_values(["year", "CD_MUN"]).to_csv(OUT, index=False)
print("saved", OUT, "| SHA-256", hashlib.sha256(OUT.read_bytes()).hexdigest())

In [ ]:
# 7. Comparison with FINN and MapBiomas Fogo (needs the repository tables; same statistics as section 3 of the manuscript)
import numpy as np
from scipy import stats
if REPO_ROOT is None:
    raise SystemExit("copy the CSV to data/external/ and run this cell inside the repository")
ev = pd.read_csv(REPO_ROOT / "data" / "derived" / "finn_sc_records_2003_2025.csv.gz",
                 usecols=["year", "sensor", "version", "AREA", "CD_MUN"])
ev = ev[(ev.sensor == "MODIS+VIIRS") & ev.version.isin(["v2.5", "v2.5.1"]) & ev.year.between(2012, 2024)]
finn_mun = ev.groupby(["year", "CD_MUN"]).AREA.sum().div(1e6).rename("finn_km2")
mb = pd.read_csv(REPO_ROOT / "data" / "external" / "mapbiomas_fogo_col4_SC_municipal_km2_2012_2024.csv")
mb = mb.set_index(["year", "CD_MUN"]).burned_km2.rename("mb_km2")
m = mcd.set_index(["year", "CD_MUN"]).burned_km2.rename("mcd_km2")
D = pd.concat([m, mb, finn_mun], axis=1).fillna(0).reset_index()
S = D.groupby("year")[["mcd_km2", "mb_km2", "finn_km2"]].sum()
S["finn_over_mcd"] = S.finn_km2 / S.mcd_km2
S["mb_over_mcd"] = S.mb_km2 / S.mcd_km2
print(S.round(2).to_string())
print("median FINN/MCD64A1 ratio:", round(S.finn_over_mcd.median(), 1), "| median MapBiomas/MCD64A1:", round(S.mb_over_mcd.median(), 2))
for a, b in (("mcd_km2", "finn_km2"), ("mcd_km2", "mb_km2")):
    r = stats.pearsonr(S[a], S[b]); rho = stats.spearmanr(S[a], S[b])
    print(f"state series {a} vs {b}: Pearson r = {r[0]:.2f} (p = {r[1]:.3f}); Spearman rho = {rho[0]:.2f} (p = {rho[1]:.3f}); n = {len(S)}")
for a, b in (("mcd_km2", "finn_km2"), ("mcd_km2", "mb_km2")):
    rr = [stats.spearmanr(g[a], g[b])[0] for _, g in D.groupby("year")]
    print(f"municipal Spearman {a} vs {b}: {min(rr):.2f} to {max(rr):.2f} across years (n = 295)")
att = pd.read_csv(REPO_ROOT / "data" / "external" / "sc_municipios_2024_atributos.csv")[["CD_MUN", "mesorregiao"]]
D = D.merge(att, on="CD_MUN")
sh = D.groupby("mesorregiao")[["mcd_km2", "mb_km2", "finn_km2"]].sum()
print("share of the Serrana, 2012-2024 (%):", (100 * sh.loc["Serrana"] / sh.sum()).round(1).to_dict())

## Next steps

1. Copy `mcd64a1_SC_municipal_km2_2012_2024.csv` to `data/external/` (if the notebook ran outside the repository) and
   record its SHA-256 in `docs/provenance.md`.
2. Fill the two passages highlighted in yellow in the manuscript (sections 2.7 and 3) and the reference to Giglio et
   al. (2018) with the statistics printed in cell 7.